<a href="https://colab.research.google.com/github/dgylayse/AkademiQ_DataScience/blob/main/AkademiQ_Data_Science_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mlflow scikit-learn pandas numpy -q
print("Kurulum tamamlandı")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

In [2]:
import mlflow
import mlflow.sklearn
import pickle
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

print("Kütüphaneler hazır")

Kütüphaneler hazır


## MLFLOW'u Dosya Tabanlı Çalıştırmak

In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("musteri-kayip-riski")
print("MLFlow hazır")

2026/07/17 16:30:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/17 16:30:02 INFO mlflow.store.db.utils: Updating database tables
2026/07/17 16:30:04 INFO mlflow.tracking.fluent: Experiment with name 'musteri-kayip-riski' does not exist. Creating a new experiment.


MLFlow hazır


## Veri Hazırlama

In [5]:
df = pd.read_csv("musteri_rfm_dt.csv", sep=";")

df['kanal_enc'] = LabelEncoder().fit_transform(df['dominant_kanal'])
df['sehir_enc'] = LabelEncoder().fit_transform(df['sehir'])

features = ['recency_gun','frequency','monetary','ort_sepet','kanal_enc','sehir_enc']
X = df[features]
y = df['kayip_riski']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

## Model yükleme

In [7]:

with open("/content/svm_model.pkl","rb") as f:
  svm = pickle.load(f)

with mlflow.start_run(run_name="svm-rbf-baseline") as run:
  y_pred = svm.predict(X_test_sc) # modelin test verisine verdiği etiket tahminleri
  y_prob = svm.predict_proba(X_test_sc)[:,1] # her müşteri için kayıp riski var olasılığı.

  acc = accuracy_score(y_test, y_pred)
  prec = precision_score(y_test, y_pred)
  rec = recall_score(y_test, y_pred)
  f1 = f1_score(y_test,y_pred)
  auc = roc_auc_score(y_test,y_prob)

  mlflow.log_param("kernel", "rbf")
  mlflow.log_param("C", 1)
  mlflow.log_param("gamma","scale")
  mlflow.log_param("test_size",0.2)

  mlflow.log_metric("accuracy",acc)
  mlflow.log_metric("precision",prec)
  mlflow.log_metric("recall",rec)
  mlflow.log_metric("f1_score",f1)
  mlflow.log_metric("roc_auc",auc)

  mlflow.log_artifact("/content/svm_model.pkl", artifact_path="models")

  run_id = run.info.run_id
  print(f"Run ID : {run_id}")
  print(f"Accuracy : {acc:.4f}")
  print(f"Precision : {prec:.4f}")
  print(f"Recall : {rec:.4f}")
  print(f"F1 Score : {f1:.4f}")
  print(f"Roc Auc : {auc:.4f}")



Run ID : a2f40cdabab24e33a5ad30d83b4d8a99
Accuracy : 0.9663
Precision : 0.9363
Recall : 1.0000
F1 Score : 0.9671
Roc Auc : 0.9907


# Tüm Run'lar Colab İçerisinde Listelenir

In [8]:
from mlflow.tracking import MlflowClient

client = MlflowClient("sqlite:///mlflow.db")

experiment = client.get_experiment_by_name("musteri-kayip-riski")
runs  = client.search_runs(experiment.experiment_id)

rows = []
for r in runs:
  rows.append({
      "Run Adı": r.info.run_name,
      "Accuracy": round(r.data.metrics.get("accuracy",0),4),
      "F1 Score": round(r.data.metrics.get("f1_score",0),4),
      "Roc Auc": round(r.data.metrics.get("roc_auc",0),4),
      "Kernel": r.data.params.get("kernel","-"),
      "C": r.data.params.get("C","-"),
  })

  print(pd.DataFrame(rows).to_string(index=False))



         Run Adı  Accuracy  F1 Score  Roc Auc Kernel C
svm-rbf-baseline    0.9663    0.9671   0.9907    rbf 1


#modeli registry 'a al

In [9]:
#Prod için hazır olan modele run açıp metrikleri kaydediyoruz.
with mlflow.start_run(run_name="svm-production-candidate"):
  y_pred = svm.predict(X_test_sc)
  y_prob = svm.predict_proba(X_test_sc)[:,1]
  mlflow.log_metric("accuracy",accuracy_score(y_test,y_pred))
  mlflow.log_metric("roc_auc",roc_auc_score(y_test,y_prob))

# Modeli registry'e taşıma

  mlflow.sklearn.log_model(
      svm, artifact_path = "svm-model", registered_model_name="musteri-kayip-svm"
  )
print("model registry'e alındı ")




2026/07/17 16:43:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


model registry'e alındı 


Successfully registered model 'musteri-kayip-svm'.
Created version '1' of model 'musteri-kayip-svm'.


#Prod etiketi verme

In [10]:
client.transition_model_version_stage(
    name = "musteri-kayip-svm",
    version = 1,
    stage = "Production" # 3 temel etiket var. Staging = henüz model testte, production = Canlıda çalışıyor, archived = model artık kullanılmıyor.
)
print("Model Production'a alındı")

Model Production'a alındı


/tmp/ipykernel_555/3067959402.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
